# Worried / calm sentiment: TF-IDF baseline and mBERT

Train on **`gold_standard_sentiment_annotated.csv`** using `post` → `sentiment_label`:
**0 = calm, 1 = worried**. These are the supplied dataset categories; labels are not
reinterpreted or regenerated. The disease vector `annotate` is never a model input.

Compare **word + character TF-IDF → balanced logistic regression** with
**multilingual BERT → a two-class classification head**. Both models use the same
deduplicated 70/15/15 split. Select thresholds and the preferred architecture on
validation worried-class F1, then evaluate the held-out test set.

## Run locally in VS Code / Jupyter

1. Open this repository and select its **`.venv/bin/python`** notebook kernel.
   The import cell prints the Python executable so you can confirm the kernel.
2. Run the setup cells. Dependencies are already installed in the project environment;
   enable `INSTALL_DEPENDENCIES` only if your selected kernel needs packages.
3. Run all cells in order. The CSV is read directly from
   `data/training_data/gold_standard_sentiment_annotated.csv`.
4. Models and reports are saved locally under `artifacts/binary_sentiment/<run_id>/`.
   The first mBERT run downloads pretrained weights; subsequent runs use the cache.
   Set `LOCAL_FILES_ONLY=True` once the model has been cached to work offline.

Device selection is automatic: **CUDA → Apple MPS → CPU**. Set
`DEVICE_PREFERENCE='cpu'` to force CPU execution. The default batch size is 4 on
Mac/CPU and 8 on CUDA, with gradient accumulation keeping an effective batch of
16 on one device. MPS and CPU use full precision. Both models run by default;
`RUN_MBERT=False` runs just the fast baseline. Full mBERT training on CPU is slow.
Restart the kernel after changing the device preference; rerun from configuration
after changing data or model settings. Do not tune on test errors.

The source has many duplicate posts and few worried examples. A 94% accuracy
can be obtained by always predicting calm on the current unique posts. Focus on
worried precision, recall, F1, and average precision. Exact normalized duplicates
are removed; the CSV has no author/thread metadata for controlling those overlaps.

Architecture references: [Google multilingual BERT](https://github.com/google-research/bert/blob/master/multilingual.md),
[scikit-learn sparse text classification](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html),
[Hugging Face Trainer](https://huggingface.co/docs/transformers/main_classes/trainer).

In [ ]:
# Optional dependency installation into the selected local notebook kernel.
import os
import subprocess
import sys

# Set before importing torch; unsupported MPS operations may fall back to CPU.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch>=2.6,<3', 'transformers>=4.57,<6', 'accelerate>=1.1,<2',
        'scikit-learn>=1.5,<2', 'pandas>=2.2,<4', 'numpy>=1.26,<3',
        'matplotlib>=3.8,<4', 'seaborn>=0.13,<1', 'joblib>=1.4,<2', 'ipykernel>=6',
    ])
    print('Dependencies installed. If packages were already imported, restart the kernel.')

In [ ]:
import hashlib
import json
import math
import random
import re
import warnings
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from uuid import uuid4

PACKAGE_NAMES = ('torch', 'transformers', 'accelerate', 'scikit-learn',
                 'pandas', 'numpy', 'matplotlib', 'seaborn', 'joblib')
try:
    PACKAGE_VERSIONS = {name: version(name) for name in PACKAGE_NAMES}
except PackageNotFoundError as exc:
    raise RuntimeError(
        'Missing notebook dependency. Enable INSTALL_DEPENDENCIES above, '
        'run that cell, then restart the kernel and rerun.'
    ) from exc

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed,
)

print('Python:', sys.executable)
print(json.dumps(PACKAGE_VERSIONS, indent=2))
sns.set_theme(style='whitegrid')

def select_local_device(preference='auto'):
    if preference not in {'auto', 'cuda', 'mps', 'cpu'}:
        raise ValueError("Device must be 'auto', 'cuda', 'mps', or 'cpu'.")
    available = {
        'cuda': torch.cuda.is_available(),
        'mps': hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(),
        'cpu': True,
    }
    if preference == 'auto':
        return torch.device(next(name for name in ('cuda', 'mps', 'cpu') if available[name]))
    if not available[preference]:
        raise RuntimeError(f'{preference} is unavailable in this kernel. Use auto or cpu.')
    return torch.device(preference)


In [ ]:
# Local paths work from the repository root or its notebooks directory.
CWD = Path.cwd()
ROOT_DIR = next(
    (p for p in [CWD, *CWD.parents] if (p / 'data' / 'training_data').is_dir()), CWD,
)
DATASET_PATH = ROOT_DIR / 'data' / 'training_data' / 'gold_standard_sentiment_annotated.csv'
OUTPUT_ROOT = ROOT_DIR / 'artifacts' / 'binary_sentiment'
DEVICE_PREFERENCE = 'auto'  # auto, cuda, mps, or cpu; restart kernel after changing.
DEVICE = select_local_device(DEVICE_PREFERENCE)
RUN_MBERT = True
LOCAL_FILES_ONLY = False
BASE_MODEL_NAME = 'google-bert/bert-base-multilingual-cased'

SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
MIN_CLASS_ROWS = 10
LABEL_NAMES = {0: 'calm', 1: 'worried'}
ID2LABEL = dict(LABEL_NAMES)
LABEL2ID = {name: label for label, name in ID2LABEL.items()}

MAX_LENGTH = 256
BATCH_SIZE = 8 if DEVICE.type == 'cuda' else 4
GRADIENT_ACCUMULATION_STEPS = 16 // BATCH_SIZE
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 2

OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / RUN_ID  # New directory per experiment; no old model is overwritten.

random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
print('Dataset:', DATASET_PATH)
print('Device:', DEVICE)
print('Batch size:', BATCH_SIZE, '| Gradient accumulation:', GRADIENT_ACCUMULATION_STEPS)
if DEVICE.type == 'cpu' and RUN_MBERT:
    warnings.warn('No local GPU is available to this kernel. Full mBERT training will run slowly on CPU.')
print('Run directory:', RUN_DIR)

## 1. Validate, deduplicate, and split

Only whitespace is normalized. Identical normalized posts are retained once;
conflicting labels stop the run. `source_row` identifies the original data row
(1-based, excluding the header). Validation happens before any output is created.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_text(text):
    return re.sub(r'\s+', ' ', text).strip()


def resolve_dataset_path(configured_path):
    if configured_path is not None:
        path = Path(configured_path).expanduser()
        if path.is_file():
            return path.resolve()
        raise FileNotFoundError(f'DATASET_PATH does not exist: {path}')
    filename = 'gold_standard_sentiment_annotated.csv'
    candidates = [ROOT_DIR / 'data' / 'training_data' / filename,
                  CWD / filename]
    matches = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f'Multiple dataset files found: {matches}. Set DATASET_PATH explicitly.')
    raise FileNotFoundError(
        f'Place {filename} in data/training_data or set DATASET_PATH to a local CSV. '
        f'Checked: {[str(p) for p in candidates]}'
    )


def validate_and_prepare_annotations(path):
    # Keep text such as "NA" literal; only blank strings count as missing here.
    raw = pd.read_csv(path, dtype=str, encoding='utf-8-sig', keep_default_na=False)
    missing = {'post', 'sentiment_label'} - set(raw.columns)
    if missing:
        raise ValueError(f'{Path(path).name}: missing columns {sorted(missing)}.')
    frame = raw[['post', 'sentiment_label']].copy()
    frame['source_row'] = np.arange(1, len(frame) + 1)
    frame['text'] = frame['post'].map(normalize_text)
    frame['label_text'] = frame['sentiment_label'].str.strip()
    bad_text = frame['text'].eq('')
    bad_label = ~frame['label_text'].isin(['0', '1'])
    if bad_text.any() or bad_label.any():
        examples = frame.loc[bad_text | bad_label, ['source_row', 'sentiment_label']].head(10)
        raise ValueError(
            f'Invalid CSV: {int(bad_text.sum())} blank posts; '
            f'{int(bad_label.sum())} missing/non-binary labels. '
            f'Fix these source rows before training: {examples.to_dict("records")}'
        )
    frame['label'] = frame['label_text'].astype(int)
    conflicts = frame.groupby('text')['label'].nunique()
    conflicts = conflicts[conflicts > 1].index
    if len(conflicts):
        examples = frame.loc[frame['text'].isin(conflicts), ['source_row', 'label']].head(20)
        raise ValueError(
            f'{len(conflicts)} posts have conflicting labels. Adjudicate source rows: '
            f'{examples.to_dict("records")}'
        )
    clean = frame.drop_duplicates('text').copy()
    clean['text_fingerprint'] = clean['text'].map(
        lambda text: hashlib.sha256(text.encode('utf-8')).hexdigest()
    )
    counts = clean['label'].value_counts().reindex([0, 1], fill_value=0)
    if (counts < MIN_CLASS_ROWS).any():
        raise ValueError(
            f'Need at least {MIN_CLASS_ROWS} unique posts in each class for splitting; '
            f'found {counts.to_dict()}.'
        )
    audit = {
        'dataset_path': str(Path(path).resolve()), 'dataset_sha256': sha256_file(path),
        'raw_rows': len(frame), 'unique_rows': len(clean),
        'duplicates_removed': len(frame) - len(clean), 'conflicting_texts': 0,
        'raw_class_counts': frame['label'].value_counts().sort_index().to_dict(),
        'unique_class_counts': counts.to_dict(),
        'normalization': 'collapse whitespace and strip; preserve casing and Unicode',
    }
    return clean[['text', 'label', 'source_row', 'text_fingerprint']].reset_index(drop=True), audit


def make_splits(frame):
    train_val, test = train_test_split(
        frame, test_size=TEST_SIZE, stratify=frame['label'], random_state=SEED,
    )
    train, validation = train_test_split(
        train_val, test_size=VAL_SIZE / (1.0 - TEST_SIZE),
        stratify=train_val['label'], random_state=SEED,
    )
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

In [ ]:
dataset_path = resolve_dataset_path(DATASET_PATH)
clean_df, data_quality_summary = validate_and_prepare_annotations(dataset_path)
print(json.dumps(data_quality_summary, indent=2))
train_df, val_df, test_df = make_splits(clean_df)
splits = {'train': train_df, 'validation': val_df, 'test': test_df}
split_manifest = pd.concat([
    frame.assign(split=name) for name, frame in splits.items()
], ignore_index=True)

assert clean_df['text'].is_unique and clean_df['text_fingerprint'].is_unique
assert len(split_manifest) == len(clean_df)
for name, frame in splits.items():
    assert set(frame['label']) == {0, 1}, f'{name} must contain both classes'
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert set(splits[left]['text']).isdisjoint(splits[right]['text'])
for original_split, repeated_split in zip(splits.values(), make_splits(clean_df)):
    assert original_split['text_fingerprint'].tolist() == repeated_split['text_fingerprint'].tolist()

split_counts = pd.DataFrame({
    name: frame['label'].value_counts().reindex([0, 1], fill_value=0)
    for name, frame in splits.items()
}).rename(index=LABEL_NAMES)
display(split_counts)
print(f"Always-calm accuracy on all unique posts: {(clean_df['label'] == 0).mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = pd.DataFrame({
    'raw': data_quality_summary['raw_class_counts'],
    'unique': data_quality_summary['unique_class_counts'],
}).rename(index=LABEL_NAMES)
counts.plot.bar(ax=axes[0], rot=0, title='Raw and deduplicated label counts')
word_lengths = train_df['text'].str.split().str.len()
sns.histplot(word_lengths, bins=40, ax=axes[1])
axes[1].set(title='Training post lengths', xlabel='Words')
plt.tight_layout()
plt.show()
display(word_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

In [ ]:
# All probabilities refer to class 1 (worried). Threshold selection sees validation only.
def metrics_at_threshold(labels, worried_probs, threshold=0.5):
    labels = np.asarray(labels, dtype=int)
    probs = np.asarray(worried_probs, dtype=float)
    if probs.shape != labels.shape or not np.isfinite(probs).all():
        raise ValueError('Expected one finite probability per label.')
    if not ((probs >= 0) & (probs <= 1)).all() or not 0 <= threshold <= 1:
        raise ValueError('Probabilities and threshold must be in [0, 1].')
    predictions = (probs >= threshold).astype(int)
    precision, recall, worried_f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', pos_label=1, zero_division=0,
    )
    return {
        'threshold': float(threshold), 'rows': int(len(labels)),
        'worried_support': int((labels == 1).sum()),
        'worried_precision': float(precision), 'worried_recall': float(recall),
        'worried_f1': float(worried_f1),
        'macro_f1': float(f1_score(labels, predictions, labels=[0, 1], average='macro', zero_division=0)),
        'average_precision': float(average_precision_score(labels, probs)),
        'accuracy': float(accuracy_score(labels, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
    }, predictions


def tune_f1_threshold(labels, probabilities):
    candidates = np.unique(np.r_[0.5, np.asarray(probabilities, dtype=float)])
    table = pd.DataFrame([
        metrics_at_threshold(labels, probabilities, threshold)[0] for threshold in candidates
    ])
    table['distance_from_0_5'] = (table['threshold'] - 0.5).abs()
    table = table.sort_values(
        ['worried_f1', 'distance_from_0_5', 'threshold'],
        ascending=[False, True, True], kind='stable',
    ).reset_index(drop=True)
    return float(table.iloc[0]['threshold']), table


def probabilities_from_logits(logits):
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.softmax(torch.as_tensor(logits, dtype=torch.float32), dim=-1).numpy()[:, 1]


def compute_metrics(prediction):
    return metrics_at_threshold(
        prediction.label_ids, probabilities_from_logits(prediction.predictions), 0.5,
    )[0]


validation_probabilities = {}
thresholds = {}
threshold_searches = {}

## 2. Train the TF-IDF baseline

Word 1–2 grams represent words and short phrases; character 3–5 grams represent
short spelling patterns. Both vocabularies and IDF weights are fitted only on
training posts. Class balancing is calculated by logistic regression from the
training labels. The saved pipeline includes both vectorizers and the classifier.

In [ ]:
baseline_model = Pipeline([
    ('features', FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 2), lowercase=False, sublinear_tf=True)),
        ('character', TfidfVectorizer(
            analyzer='char', ngram_range=(3, 5), lowercase=False, sublinear_tf=True,
        )),
    ])),
    ('classifier', LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=2000, random_state=SEED,
    )),
])
baseline_model.fit(train_df['text'], train_df['label'])
assert baseline_model.named_steps['classifier'].classes_.tolist() == [0, 1]
validation_probabilities['tfidf_logreg'] = baseline_model.predict_proba(val_df['text'])[:, 1]
thresholds['tfidf_logreg'], threshold_searches['tfidf_logreg'] = tune_f1_threshold(
    val_df['label'], validation_probabilities['tfidf_logreg'],
)
display(threshold_searches['tfidf_logreg'].head(5))

## 3. Fine-tune mBERT

Tokenize with a 256-token limit, including special tokens, and pad only to the
longest sequence in each batch. Long posts are truncated; rates are reported
below. Sequence length is fixed before looking at test performance.

The loss uses inverse-frequency weights from the training split. Checkpoints are
selected by **validation worried F1 at threshold 0.5**; the probability threshold
is tuned separately after loading the best checkpoint. The weighted loss follows
the Trainer's gradient-accumulation contract. Apple MPS and CPU runs use full precision; CUDA
runs use FP16. Checkpoints and reports are local files; no tracking service is used.

In [ ]:
class PostDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer, max_length):
        self.encodings = tokenizer(
            frame['text'].tolist(), truncation=True, max_length=max_length, padding=False,
        )
        self.labels = frame['label'].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {**{key: values[index] for key, values in self.encodings.items()},
                'labels': self.labels[index]}


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        # Loss is a microbatch mean; let Trainer scale for gradient accumulation.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        model_inputs = dict(inputs)
        labels = model_inputs.pop('labels')
        outputs = model(**model_inputs)
        logits = outputs.logits.float()
        loss = torch.nn.functional.cross_entropy(
            logits, labels, weight=self.class_weights.to(logits.device),
        )
        return (loss, outputs) if return_outputs else loss


def make_training_arguments(output_dir, training_rows):
    steps_per_epoch = math.ceil(math.ceil(training_rows / BATCH_SIZE) / GRADIENT_ACCUMULATION_STEPS)
    kwargs = dict(
        output_dir=str(output_dir), eval_strategy='epoch', save_strategy='epoch',
        logging_strategy='epoch', learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS, weight_decay=WEIGHT_DECAY,
        warmup_steps=math.ceil(steps_per_epoch * NUM_EPOCHS * WARMUP_RATIO),
        load_best_model_at_end=True, metric_for_best_model='eval_worried_f1',
        greater_is_better=True, save_total_limit=2, report_to=[],
        seed=SEED, data_seed=SEED, use_cpu=DEVICE.type == 'cpu',
        fp16=DEVICE.type == 'cuda', bf16=False, optim='adamw_torch',
        dataloader_num_workers=0, dataloader_pin_memory=DEVICE.type == 'cuda',
    )
    return TrainingArguments(**kwargs)


def token_length_audit(frame, tokenizer):
    lengths = []
    for start in range(0, len(frame), 256):
        encoded = tokenizer(
            frame['text'].iloc[start:start + 256].tolist(),
            truncation=False, padding=False, return_length=True, verbose=False,
        )
        lengths.extend(encoded['length'])
    lengths = np.asarray(lengths)
    return {'rows': len(frame), 'token_length_p95': float(np.quantile(lengths, 0.95)),
            'truncated_posts': int((lengths > MAX_LENGTH).sum()),
            'truncation_rate': float((lengths > MAX_LENGTH).mean()), 'max_length': MAX_LENGTH}

In [ ]:
token_audits = {}
train_counts = train_df['label'].value_counts().reindex([0, 1]).astype(float)
class_weights = torch.tensor(len(train_df) / (2.0 * train_counts.to_numpy()), dtype=torch.float32)
print('Training-only class weights:', dict(zip(LABEL_NAMES.values(), class_weights.tolist())))

if RUN_MBERT:
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            BASE_MODEL_NAME, use_fast=True, local_files_only=LOCAL_FILES_ONLY,
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
            local_files_only=LOCAL_FILES_ONLY,
        )
    except OSError as exc:
        raise RuntimeError(
            'Could not load mBERT. Enable internet for the first download, or use '
            'LOCAL_FILES_ONLY=True with an existing cache. For a CPU baseline run, '
            'set RUN_MBERT=False and rerun from configuration.'
        ) from exc

    token_audits = {name: token_length_audit(frame, tokenizer) for name, frame in splits.items()}
    display(pd.DataFrame(token_audits).T)
    train_ds = PostDataset(train_df, tokenizer, MAX_LENGTH)
    val_ds = PostDataset(val_df, tokenizer, MAX_LENGTH)
    test_ds = PostDataset(test_df, tokenizer, MAX_LENGTH)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    training_args = make_training_arguments(RUN_DIR / 'mbert' / 'checkpoints', len(train_df))
    if training_args.device.type != DEVICE.type:
        raise RuntimeError('Trainer device differs from DEVICE. Restart the kernel and rerun.')
    trainer = WeightedLossTrainer(
        model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=data_collator, processing_class=tokenizer, compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    train_result = trainer.train()
    val_output = trainer.predict(val_ds, metric_key_prefix='validation')
    assert np.array_equal(val_output.label_ids, val_df['label'].to_numpy())
    validation_probabilities['mbert'] = probabilities_from_logits(val_output.predictions)
    thresholds['mbert'], threshold_searches['mbert'] = tune_f1_threshold(
        val_df['label'], validation_probabilities['mbert'],
    )
    display(threshold_searches['mbert'].head(5))
else:
    print('Baseline-only run: mBERT training and comparison are skipped.')

## 4. Freeze model and threshold choices using validation

Rank trained models by validation worried F1 at their tuned thresholds. Exact
model-score ties favor logistic regression. Threshold-score ties favor the value
nearest 0.5, then the lower threshold. The always-calm predictor is a reference,
not a candidate architecture. No test predictions have been used for selection.

In [ ]:
validation_rows = []
for name, probabilities in validation_probabilities.items():
    metrics, _ = metrics_at_threshold(val_df['label'], probabilities, thresholds[name])
    validation_rows.append({'model': name, **metrics})
validation_table = pd.DataFrame(validation_rows)
validation_table['tie_priority'] = validation_table['model'].map({'tfidf_logreg': 0, 'mbert': 1})
validation_table = validation_table.sort_values(
    ['worried_f1', 'tie_priority'], ascending=[False, True], kind='stable',
).drop(columns='tie_priority').reset_index(drop=True)
preferred_model = validation_table.iloc[0]['model']
frozen_thresholds = dict(thresholds)
selection_payload = {
    'preferred_model': preferred_model,
    'comparison_complete': RUN_MBERT,
    'selection_metric': 'validation worried F1 at validation-tuned threshold',
    'model_tie_break': 'tfidf_logreg',
    'threshold_tie_break': 'closest to 0.5, then lower threshold',
    'thresholds': frozen_thresholds,
}
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'selection.json').write_text(json.dumps(selection_payload, indent=2), encoding='utf-8')
display(validation_table)
print('Preferred model (validation only):', preferred_model)

In [ ]:
# Freeze choices above BEFORE evaluating test. Do not select models using this table.
test_probabilities = {'tfidf_logreg': baseline_model.predict_proba(test_df['text'])[:, 1]}
if RUN_MBERT:
    test_output = trainer.predict(test_ds, metric_key_prefix='test')
    assert np.array_equal(test_output.label_ids, test_df['label'].to_numpy())
    test_probabilities['mbert'] = probabilities_from_logits(test_output.predictions)
test_probabilities['always_calm'] = np.zeros(len(test_df))

test_rows, test_predictions, test_confusion_matrices, test_reports = [], {}, {}, {}
for name, probabilities in test_probabilities.items():
    threshold = frozen_thresholds.get(name, 0.5)
    metrics, predictions = metrics_at_threshold(test_df['label'], probabilities, threshold)
    test_rows.append({'model': name, **metrics})
    test_reports[name] = classification_report(
        test_df['label'], predictions, labels=[0, 1], target_names=['calm', 'worried'],
        output_dict=True, zero_division=0,
    )
    frame = test_df.copy()
    frame['true_label_name'] = frame['label'].map(LABEL_NAMES)
    frame['worried_probability'] = probabilities
    frame['predicted_label'] = predictions
    frame['predicted_label_name'] = frame['predicted_label'].map(LABEL_NAMES)
    frame['threshold_used'] = threshold
    frame['error_type'] = np.select(
        [(frame['label'] == 0) & (predictions == 1),
         (frame['label'] == 1) & (predictions == 0)],
        ['false_positive', 'false_negative'], default='correct',
    )
    test_predictions[name] = frame
    test_confusion_matrices[name] = pd.DataFrame(
        confusion_matrix(frame['label'], predictions, labels=[0, 1]),
        index=['true_calm', 'true_worried'], columns=['pred_calm', 'pred_worried'],
    )

test_table = pd.DataFrame(test_rows).set_index('model')
display(test_table)
print('Preferred architecture remains:', preferred_model)
print(f"Test worried support: {int((test_df['label'] == 1).sum())}. "
      'Small score differences may be unstable with this few worried examples.')
for name in validation_probabilities:
    print(name, '— false positives')
    display(test_predictions[name].query("error_type == 'false_positive'")
            .sort_values('worried_probability', ascending=False).head(5))
    print(name, '— false negatives')
    display(test_predictions[name].query("error_type == 'false_negative'")
            .sort_values('worried_probability').head(5))

In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def write_json(path, payload):
    Path(path).write_text(json.dumps(json_ready(payload), indent=2, allow_nan=False), encoding='utf-8')


common_config = {
    'run_id': RUN_ID, 'dataset_path': dataset_path,
    'dataset_sha256': data_quality_summary['dataset_sha256'],
    'seed': SEED, 'split': {'train': 1 - VAL_SIZE - TEST_SIZE, 'validation': VAL_SIZE, 'test': TEST_SIZE},
    'input_column': 'post', 'target_column': 'sentiment_label',
    'text_normalization': data_quality_summary['normalization'],
    'versions': PACKAGE_VERSIONS, 'device': str(DEVICE), 'run_mbert': RUN_MBERT,
}
model_dirs = {}
manifest_columns = ['text_fingerprint', 'source_row', 'label', 'split']
for name in validation_probabilities:
    model_dir = RUN_DIR / name
    report_dir = model_dir / 'reports'
    report_dir.mkdir(parents=True, exist_ok=True)
    model_dirs[name] = model_dir
    config = {**common_config, 'model_key': name}
    if name == 'tfidf_logreg':
        joblib.dump(baseline_model, model_dir / 'pipeline.joblib')
        config.update(word_ngram_range=[1, 2], char_ngram_range=[3, 5],
                      lowercase=False, sublinear_tf=True, class_weight='balanced',
                      C=1.0, max_iter=2000)
    else:
        trainer.save_model(str(model_dir / 'best_model'))
        tokenizer.save_pretrained(model_dir / 'best_model')
        config.update(
            base_model_name=BASE_MODEL_NAME, local_files_only=LOCAL_FILES_ONLY,
            model_commit=getattr(model.config, '_commit_hash', None),
            max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO, warmup_steps=training_args.warmup_steps,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            class_weights=class_weights.tolist(), fp16=training_args.fp16,
            best_checkpoint=trainer.state.best_model_checkpoint,
            checkpoint_metric='validation worried F1 at threshold 0.5',
        )
        write_json(report_dir / 'token_lengths.json', token_audits)
        write_json(report_dir / 'training_metrics.json', train_result.metrics)
        log_df = pd.DataFrame(trainer.state.log_history)
        log_df.to_csv(report_dir / 'training_log.csv', index=False)
        fig, ax = plt.subplots(figsize=(8, 4))
        for field, label in [('loss', 'train'), ('eval_loss', 'validation')]:
            if field in log_df:
                history = log_df.dropna(subset=[field])
                ax.plot(history['epoch'], history[field], marker='o', label=label)
        ax.set(xlabel='Epoch', ylabel='Weighted loss', title='mBERT training history')
        ax.legend()
        fig.tight_layout()
        fig.savefig(report_dir / 'training_loss.png', dpi=150)
        plt.show()

    write_json(model_dir / 'run_config.json', config)
    write_json(model_dir / 'label_mapping.json', {'labels': LABEL_NAMES, 'positive_class': 1})
    write_json(model_dir / 'threshold.json', {'worried_threshold': frozen_thresholds[name]})
    write_json(report_dir / 'data_quality.json', data_quality_summary)
    write_json(report_dir / 'metrics.json', {
        'validation': validation_table.set_index('model').loc[name].to_dict(),
        'test': test_table.loc[name].to_dict(),
        'classification_report': test_reports[name], 'selection': selection_payload,
    })
    split_manifest[manifest_columns].to_csv(report_dir / 'split_manifest.csv', index=False)
    threshold_searches[name].to_csv(report_dir / 'validation_threshold_search.csv', index=False)
    test_predictions[name].to_csv(report_dir / 'test_predictions.csv', index=False)
    test_predictions[name].query("error_type != 'correct'").to_csv(report_dir / 'errors.csv', index=False)
    test_confusion_matrices[name].to_csv(report_dir / 'confusion_matrix.csv')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.heatmap(test_confusion_matrices[name], annot=True, fmt='d', cbar=False, cmap='Blues', ax=axes[0])
    axes[0].set_title(f'{name}: test confusion matrix')
    for split_name, labels, probabilities in [
        ('validation', val_df['label'], validation_probabilities[name]),
        ('test', test_df['label'], test_probabilities[name]),
    ]:
        precision, recall, _ = precision_recall_curve(labels, probabilities)
        axes[1].plot(recall, precision, label=split_name)
    axes[1].set(xlabel='Worried recall', ylabel='Worried precision', xlim=(0, 1), ylim=(0, 1))
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(report_dir / 'evaluation.png', dpi=150)
    plt.show()

reference_dir = RUN_DIR / 'always_calm'
reference_dir.mkdir(exist_ok=True)
write_json(reference_dir / 'metrics.json', test_table.loc['always_calm'].to_dict())
test_confusion_matrices['always_calm'].to_csv(reference_dir / 'confusion_matrix.csv')
display(test_confusion_matrices['always_calm'])
validation_table.to_csv(RUN_DIR / 'validation_comparison.csv', index=False)
test_table.to_csv(RUN_DIR / 'test_comparison.csv')
write_json(RUN_DIR / 'selection.json', selection_payload)
print('Saved run:', RUN_DIR)

## 5. Reload saved artifacts for notebook inference

The loader reads the saved label mapping, threshold, normalization contract, and
model configuration. It does not use training-time thresholds from memory. For a
new session, run dependencies/imports and this loader definition, then pass the
saved model directory to `load_sentiment_predictor`:

```python
predict = load_sentiment_predictor("/path/to/HealthPH-Plus/artifacts/binary_sentiment/<run_id>/mbert")
predict("Kinakabahan ako, my fever is not going away.")
```

In [ ]:
def load_sentiment_predictor(model_dir, device=None):
    model_dir = Path(model_dir)
    config = json.loads((model_dir / 'run_config.json').read_text())
    labels = {int(key): value for key, value in
              json.loads((model_dir / 'label_mapping.json').read_text())['labels'].items()}
    threshold = float(json.loads((model_dir / 'threshold.json').read_text())['worried_threshold'])
    if labels != {0: 'calm', 1: 'worried'} or not 0 <= threshold <= 1:
        raise ValueError('Saved label mapping or threshold does not match this notebook.')
    expected_normalization = 'collapse whitespace and strip; preserve casing and Unicode'
    if config['text_normalization'] != expected_normalization:
        raise ValueError('Unsupported saved normalization contract.')
    target_device = select_local_device(str(device) if device is not None else 'auto')
    if config['model_key'] == 'tfidf_logreg':
        saved_pipeline = joblib.load(model_dir / 'pipeline.joblib')
        positive_column = list(saved_pipeline.classes_).index(1)
    elif config['model_key'] == 'mbert':
        saved_tokenizer = AutoTokenizer.from_pretrained(model_dir / 'best_model', local_files_only=True)
        saved_model = AutoModelForSequenceClassification.from_pretrained(
            model_dir / 'best_model', local_files_only=True,
        ).to(target_device)
        if saved_model.config.id2label != labels:
            raise ValueError('Transformer label mapping differs from the saved label mapping.')
        saved_model.eval()
    else:
        raise ValueError(f"Unknown saved model type: {config['model_key']}")

    def predict(text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError('text must be a non-empty string')
        text = re.sub(r'\s+', ' ', text).strip()
        if config['model_key'] == 'tfidf_logreg':
            probability = float(saved_pipeline.predict_proba([text])[0, positive_column])
        else:
            encoded = saved_tokenizer(
                text, truncation=True, max_length=config['max_length'], return_tensors='pt',
            ).to(target_device)
            with torch.inference_mode():
                probability = float(torch.softmax(saved_model(**encoded).logits.float(), dim=-1)[0, 1].cpu())
        label = int(probability >= threshold)
        return {'predicted_label': label, 'predicted_label_name': labels[label],
                'worried_probability': probability, 'threshold_used': threshold}

    return predict

In [1]:
# Compare reloaded models against prior validation predictions, not a second test pass.
if RUN_MBERT:
    trainer.model.to('cpu')  # Move trained weights off the GPU before loading the saved copy.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for name, model_dir in model_dirs.items():
    reloaded_predict = load_sentiment_predictor(model_dir, device=DEVICE)
    for index in range(min(8, len(val_df))):
        result = reloaded_predict(val_df.iloc[index]['text'])
        assert set(result) == {'predicted_label', 'predicted_label_name',
                               'worried_probability', 'threshold_used'}
        assert 0 <= result['worried_probability'] <= 1
        assert result['predicted_label_name'] == LABEL_NAMES[result['predicted_label']]
        assert result['threshold_used'] == frozen_thresholds[name]
        # CUDA mixed precision vs reloaded FP32 can produce small probability differences.
        np.testing.assert_allclose(
            result['worried_probability'], validation_probabilities[name][index],
            atol=2e-3 if name == 'mbert' else 1e-10, rtol=2e-3 if name == 'mbert' else 1e-10,
        )
    for invalid in ['', '   ', None]:
        try:
            reloaded_predict(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError('Inference must reject blank/non-string inputs.')
    del reloaded_predict

    for filename in ['run_config.json', 'label_mapping.json', 'threshold.json',
                     'reports/metrics.json', 'reports/split_manifest.csv',
                     'reports/test_predictions.csv', 'reports/confusion_matrix.csv']:
        assert (model_dir / filename).is_file(), filename
assert data_quality_summary['conflicting_texts'] == 0
assert set(validation_probabilities) == ({'tfidf_logreg', 'mbert'} if RUN_MBERT else {'tfidf_logreg'})
assert preferred_model == validation_table.iloc[0]['model']
for name, search in threshold_searches.items():
    assert search.iloc[0]['worried_f1'] == search['worried_f1'].max()
assert np.allclose(class_weights.numpy(), len(train_df) / (2 * train_counts.to_numpy()))
print('Data, split, artifact, and reload acceptance checks passed.')

predict_binary_sentiment = load_sentiment_predictor(model_dirs[preferred_model], device=DEVICE)
for example in [
    'I am worried that my fever and cough are getting worse.',
    'Buti na lang okay na siya ngayon.',
    'Kinakabahan ako, my fever is not going away.',
]:
    print(example)
    print(json.dumps(predict_binary_sentiment(example), indent=2))

NameError: name 'RUN_MBERT' is not defined